# 1. Image Segmentation

In [ ]:
import os
from rembg import remove
from PIL import Image
import numpy as np

def segment_dataset(input_folder, output_folder):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    for root, dirs, files in os.walk(input_folder):
        for file in files:
            if file.endswith(('.jpg', '.png', '.jpeg')):
                input_path = os.path.join(root, file)
                
                # 1. Open image
                input_img = Image.open(input_path)
                
                # 2. Remove background (returns image with alpha channel)
                output_img = remove(input_img)
                
                # 3. Create a background (e.g., solid white or black)
                
                canvas = Image.new("RGB", output_img.size, (0, 0, 0)) # Black bg
                canvas.paste(output_img, mask=output_img.split()[3]) # Use alpha as mask
                
                # 4. Save to new directory structure
                rel_path = os.path.relpath(root, input_folder)
                out_dir = os.path.join(output_folder, rel_path)
                if not os.path.exists(out_dir): os.makedirs(out_dir)
                
                canvas.save(os.path.join(out_dir, file))
                
#segment_dataset('../data/train', '../data/train_segmented')
segment_dataset('PATH', 'PATH')

# 2. Data Cleaning
Some of the segmented data led to an "almost blank" samples, the following section rejects any sample with less than 10% of non-black pixels.

In [ ]:
import cv2
import os
import shutil
from pathlib import Path
from tqdm import tqdm
import numpy as np


def cleanup_segmented_data(root_dir, threshold=0.05, move_to="PATH"):
    """
    root_dir: The path to your segmented images (e.g., 'data/train')
    threshold: Minimum percentage of non-black pixels (0.05 = 5%)
    move_to: Folder where 'blank' images will be moved
    """
    root_path = Path(root_dir)
    rejected_path = Path(move_to)
    
    # Supported extensions
    exts = ['.jpg', '.jpeg', '.png', '.bmp']
    
    print(f"--- Scanning {root_dir} ---")
    files = [f for f in root_path.rglob('*') if f.suffix.lower() in exts]
    
    count_removed = 0

    for file_path in tqdm(files):
        # 1. Read image in grayscale
        img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            continue
            
        # 2. Calculate what percentage of pixels are NOT black (0)
        # We use > 5 here because 'rembg' sometimes leaves very dark noise
        non_black_pixels = np.count_nonzero(img > 5)
        total_pixels = img.size
        coverage = non_black_pixels / total_pixels
        
        # 3. If coverage is below threshold, move it
        if coverage < threshold:
            # Maintain class folder structure in the rejected folder
            relative_path = file_path.relative_to(root_path)
            destination = rejected_path / relative_path
            
            # Create subfolders if they don't exist
            destination.parent.mkdir(parents=True, exist_ok=True)
            
            # Move the file
            shutil.move(str(file_path), str(destination))
            count_removed += 1

    print(f"\nCleanup Complete!")
    print(f"Total images checked: {len(files)}")
    print(f"Images moved to '{move_to}': {count_removed}")
    print(f"Percentage removed: {(count_removed/len(files))*100:.2f}%")

cleanup_segmented_data('PATH', threshold=0.05)

In [ ]:
# Quick audit — run this on your rejected folder
from collections import Counter
from pathlib import Path

rejected = Path('PATH')
class_counts = Counter(p.parent.name for p in rejected.rglob('*') if p.is_file())
for cls, count in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"{cls:35s}: {count}")